In [ ]:
import os

# Base directory — goes up one level from notebooks/ to project root
# Set base directory relative to notebook location
BASE_DIR = os.path.dirname(os.path.abspath('01_data_collection.ipynb'))
RAW_DIR = os.path.join(BASE_DIR, 'data', 'raw')
CLEAN_DIR = os.path.join(BASE_DIR, 'data', 'cleaned')
FIG_DIR = os.path.join(BASE_DIR, 'outputs', 'figures')

# Confirm paths exist
for path in [RAW_DIR, CLEAN_DIR, FIG_DIR]:
    os.makedirs(path, exist_ok=True)

print("RAW_DIR:", RAW_DIR)
print("CLEAN_DIR:", CLEAN_DIR)
print("FIG_DIR:", FIG_DIR)

In [ ]:
pip install pytrends pandas matplotlib seaborn statsmodels requests

In [ ]:
from pytrends.request import TrendReq
import pandas as pd
import matplotlib.pyplot as plt

pytrends = TrendReq(hl='en-US', tz=360)

# Pull 1: Chinese EV brands vs Tesla
kw_list_1 = ["Tesla", "BYD electric", "NIO car", "Rivian", "Lucid Motors"]
pytrends.build_payload(kw_list_1, timeframe='2020-01-01 2024-12-31', geo='US')
df_1 = pytrends.interest_over_time().drop(columns='isPartial')

# Pull 2: Demand intent signals
kw_list_2 = ["buy Tesla", "Tesla Model Y", "electric car tax credit", "EV incentive", "Chinese EV tariff"]
pytrends.build_payload(kw_list_2, timeframe='2020-01-01 2024-12-31', geo='US')
df_2 = pytrends.interest_over_time().drop(columns='isPartial')

# Pull 3: FSD events
kw_list_3 = ["Tesla FSD", "Tesla self driving", "Tesla autopilot", "full self driving"]
pytrends.build_payload(kw_list_3, timeframe='2020-01-01 2024-12-31', geo='US')
df_3 = pytrends.interest_over_time().drop(columns='isPartial')

# Save all three
df_1.to_csv('../data/raw/trends_competition.csv')
df_2.to_csv('../data/raw/trends_demand.csv')
df_3.to_csv('../data/raw/trends_fsd.csv')

print("Pull 1:"); print(df_1.tail())
print("Pull 2:"); print(df_2.tail())
print("Pull 3:"); print(df_3.tail())

In [ ]:
import pandas as pd

events = {
    "COVID Recovery": "2021-01-01",
    "IRA Signed": "2022-08-16",
    "Biden 100% Tariff Announced": "2024-05-14",
    "FSD v12 Release": "2024-03-01",
    "FSD Unsupervised Launch": "2024-10-01"
}

events_df = pd.DataFrame(list(events.items()), columns=['event', 'date'])
events_df['date'] = pd.to_datetime(events_df['date'])
events_df.to_csv('../data/raw/events.csv', index=False)
print(events_df)

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

df_1 = pd.read_csv('../data/raw/trends_competition.csv', index_col=0, parse_dates=True)
events_df = pd.read_csv('../data/raw/events.csv', parse_dates=['date'])

fig, ax = plt.subplots(figsize=(14, 6))

for col in df_1.columns:
    ax.plot(df_1.index, df_1[col], label=col)

# Add event lines
for _, row in events_df.iterrows():
    ax.axvline(x=row['date'], color='gray', linestyle='--', alpha=0.7)
    ax.text(row['date'], ax.get_ylim()[1]*0.9, row['event'], 
            rotation=90, fontsize=8, color='gray')

ax.set_title('US Search Interest: Tesla vs EV Competitors (2020-2024)')
ax.set_xlabel('Date')
ax.set_ylabel('Search Interest (Google Trends Index)')
ax.legend()
plt.tight_layout()
plt.savefig('../outputs/figures/competition_trends.png', dpi=150)
plt.show()

In [ ]:
import requests
import pandas as pd
import os

EIA_KEY = "YOUR_EIA_KEY_HERE"  # Get free key at eia.gov/opendata

url = "https://api.eia.gov/v2/petroleum/pri/gnd/data/"
params = {
    "api_key": EIA_KEY,
    "frequency": "monthly",
    "data[0]": "value",
    "start": "2020-01",
    "end": "2024-12",
    "sort[0][column]": "period",
    "sort[0][direction]": "asc",
    "offset": 0,
    "length": 5000
}

response = requests.get(url, params=params)
data = response.json()

df_gas = pd.DataFrame(data['response']['data'])
print(df_gas.shape)
print(df_gas.head())

df_gas.to_csv(os.path.join(RAW_DIR, 'eia_gas_prices.csv'), index=False)
print("Gas prices saved.")

In [ ]:
import requests
import pandas as pd
import os

NREL_KEY = "YOUR_NREL_KEY_HERE"  # Get free key at developer.nrel.gov

# Pass ev_network directly in the URL string
url = f"https://developer.nrel.gov/api/alt-fuel-stations/v1.json?api_key={NREL_KEY}&fuel_type=ELEC&ev_network=Tesla&country=US&limit=200&offset=0"

response = requests.get(url)
data = response.json()

print("Total results:", data.get('total_results'))
df_test = pd.DataFrame(data['fuel_stations'])
print(df_test['ev_network'].value_counts())
print(df_test[['station_name', 'city', 'state', 'open_date']].head())

In [ ]:
import requests
import pandas as pd
import os

NREL_KEY = "YOUR_NREL_KEY_HERE"  # Get free key at developer.nrel.gov

all_stations = []
offset = 0
limit = 200
total = 3079

while offset < total:
    url = f"https://developer.nrel.gov/api/alt-fuel-stations/v1.json?api_key={NREL_KEY}&fuel_type=ELEC&ev_network=Tesla&country=US&limit={limit}&offset={offset}"
    
    response = requests.get(url)
    data = response.json()
    stations = data.get('fuel_stations', [])
    
    if not stations:
        break
        
    all_stations.extend(stations)
    print(f"Fetched {len(all_stations)} of {total} stations...")
    offset += limit

df_stations = pd.DataFrame(all_stations)
print(f"\nDone. Shape: {df_stations.shape}")
print(df_stations[['station_name', 'city', 'state', 'open_date']].head())

df_stations.to_csv(os.path.join(RAW_DIR, 'afdc_superchargers.csv'), index=False)
print("Supercharger data saved.")

In [ ]:
import pandas as pd
import os

# Load California data
df_ca = pd.read_excel(os.path.join(RAW_DIR, 'Vehicle_Population_Last_updated_04-28-2026_ada.xlsx'))

print("Columns:", df_ca.columns.tolist())
print("Shape:", df_ca.shape)
print("Data Year range:", df_ca['Data Year'].min(), "to", df_ca['Data Year'].max())
print("Fuel types:", df_ca['Dashboard Fuel Type Group'].unique())
print("Sample Tesla rows:")
print(df_ca[df_ca['Make'] == 'Tesla'].head(10))

In [ ]:
# Filter to BEV only
df_ca_bev = df_ca[df_ca['Dashboard Fuel Type Group'] == 'Battery Electric (BEV)'].copy()

# Flag Tesla
df_ca_bev['is_tesla'] = df_ca_bev['Make'] == 'Tesla'

# Aggregate to state level by year
df_ca_agg = df_ca_bev.groupby(['Data Year', 'is_tesla'])['Number of Vehicles'].sum().reset_index()

# Pivot
df_ca_pivot = df_ca_agg.pivot_table(
    index='Data Year', columns='is_tesla', values='Number of Vehicles', aggfunc='sum'
).reset_index()
df_ca_pivot.columns = ['year', 'non_tesla_bev', 'tesla_bev']
df_ca_pivot['total_bev'] = df_ca_pivot['tesla_bev'] + df_ca_pivot['non_tesla_bev']
df_ca_pivot['tesla_share'] = df_ca_pivot['tesla_bev'] / df_ca_pivot['total_bev']

# Assign to Q4 of each year to match panel structure
df_ca_pivot['quarter'] = df_ca_pivot['year'].astype(str) + 'Q4'
df_ca_pivot['state'] = 'CA'

# Filter to 2020 onwards
df_ca_panel = df_ca_pivot[df_ca_pivot['year'] >= 2020][['state', 'quarter', 'non_tesla_bev', 'tesla_bev', 'total_bev', 'tesla_share']]

print(df_ca_panel)

In [ ]:
# Load existing clean panel
df_existing = pd.read_csv(os.path.join(CLEAN_DIR, 'ev_registrations_clean.csv'))

# Stack California
df_ca_panel['quarter'] = df_ca_panel['quarter'].astype(str)
df_combined = pd.concat([df_existing, df_ca_panel], ignore_index=True)

print("Original panel shape:", df_existing.shape)
print("Combined panel shape:", df_combined.shape)
print("\nStates in combined panel:", sorted(df_combined['state'].unique()))

# Save
df_combined.to_csv(os.path.join(CLEAN_DIR, 'ev_registrations_with_ca.csv'), index=False)
print("Saved.")

In [ ]:
kw_list_musk = ["Elon Musk Tesla", "boycott Tesla", "Tesla Elon", "Musk Twitter", "DOGE Tesla"]
pytrends.build_payload(kw_list_musk, timeframe='2020-01-01 2024-12-31', geo='US')
df_musk = pytrends.interest_over_time().drop(columns='isPartial')
df_musk.to_csv(os.path.join(RAW_DIR, 'trends_musk.csv'))
print(df_musk.tail())

In [ ]:
import time
from pytrends.request import TrendReq
import pandas as pd
import os

RAW_DIR = "ev_project/data/raw"
os.makedirs(RAW_DIR, exist_ok=True)

pytrends = TrendReq(
    hl='en-US',
    tz=360,
    timeout=(10, 25),
    retries=2,
    backoff_factor=0.1
)

def pull_trends(keywords, filename, label, sleep_time=30):
    try:
        pytrends.build_payload(
            keywords,
            timeframe='2020-01-01 2024-12-31',
            geo='US'
        )

        df = pytrends.interest_over_time()

        if 'isPartial' in df.columns:
            df = df.drop(columns='isPartial')

        df.to_csv(os.path.join(RAW_DIR, filename))

        print(f"\n{label} pull:")
        print(df.describe())

        print("\nNon-zero weeks per term:")
        print((df > 0).sum())

        print("\nMean values:")
        print(df.mean())

        time.sleep(sleep_time)
        return df

    except Exception as e:
        print(f"\nFAILED: {label}")
        print(e)
        time.sleep(60)
        return pd.DataFrame()


df_byd = pull_trends(
    ["BYD", "BYD car", "BYD electric car", "BYD EV", "BYD vehicle"],
    "trends_byd_variations.csv",
    "BYD variations"
)

df_nio = pull_trends(
    ["NIO", "NIO car", "NIO electric", "NIO EV", "NIO vehicle"],
    "trends_nio_variations.csv",
    "NIO variations"
)

df_chinese = pull_trends(
    ["Chinese electric car", "Chinese EV", "China EV", "BYD USA", "Chinese car"],
    "trends_chinese_ev_broad.csv",
    "Broad Chinese EV terms"
)

df_compare = pull_trends(
    ["BYD", "Tesla", "Rivian", "NIO", "XPENG"],
    "trends_brands_comparison.csv",
    "Direct brand comparison"
)

In [ ]:
pip install --upgrade pytrends urllib3 requests